# EDA — б/у смартфоны и ноутбуки

Цель: понять структуру данных, распределения, выбросы и зависимости признаков с ценой.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing.clean import clean_all
from src.preprocessing.features import build_features

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

## 1. Загрузка и первичный осмотр

In [ ]:
df_raw_phones = pd.read_csv(ROOT / 'data/raw/used_device_data.csv')
df_raw_laptops = pd.read_csv(ROOT / 'data/raw/laptop_price.csv', encoding='latin-1')
print(f'Смартфоны — shape: {df_raw_phones.shape}')
print(f'Ноутбуки  — shape: {df_raw_laptops.shape}')

In [ ]:
print('=== Смартфоны ===')
print(df_raw_phones.dtypes)
df_raw_phones.head(3)

In [ ]:
print('=== Ноутбуки ===')
print(df_raw_laptops.dtypes)
df_raw_laptops.head(3)

## 2. Пропуски в сырых данных

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_raw_phones.isnull().mean().sort_values(ascending=False).plot(kind='bar', ax=axes[0], title='Пропуски — смартфоны')
df_raw_laptops.isnull().mean().sort_values(ascending=False).plot(kind='bar', ax=axes[1], title='Пропуски — ноутбуки')
plt.tight_layout()

## 3. Объединённый датасет после очистки

In [ ]:
df = build_features(clean_all())
print(f'Shape после очистки: {df.shape}')
print(df['device_type'].value_counts())
df.describe()

## 4. Целевая переменная — цена

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['price'].hist(bins=60, ax=axes[0])
axes[0].set_title('Распределение цены (руб.)')
axes[0].set_xlabel('Цена')
df['log_price'].hist(bins=60, ax=axes[1])
axes[1].set_title('log(цена)')
plt.tight_layout()

## 5. Цена по типу устройства

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x='device_type', y='price', ax=axes[0])
axes[0].set_title('Цена по типу устройства')
df.groupby('device_type')['price'].median().plot(kind='bar', ax=axes[1])
axes[1].set_title('Медианная цена')
plt.tight_layout()

## 6. Топ брендов по количеству и медианной цене

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['brand'].value_counts().head(15).plot(kind='bar', ax=axes[0])
axes[0].set_title('Топ-15 брендов по количеству')
axes[0].tick_params(axis='x', rotation=45)
df.groupby('brand')['price'].median().sort_values(ascending=False).head(15).plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Топ-15 брендов по медианной цене')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()

## 7. Зависимость числовых признаков с ценой

In [ ]:
num_cols = ['storage_gb', 'ram_gb', 'screen_inch', 'condition_enc']
existing = [c for c in num_cols if c in df.columns]
fig, axes = plt.subplots(1, len(existing), figsize=(16, 4))
for ax, col in zip(axes, existing):
    ax.scatter(df[col], df['price'], alpha=0.15, s=8)
    ax.set_xlabel(col)
    ax.set_ylabel('price')
    ax.set_title(col)
plt.tight_layout()

## 8. Корреляционная матрица

In [ ]:
corr_cols = ['price', 'storage_gb', 'ram_gb', 'screen_inch', 'condition_enc',
             'is_laptop', 'age_years', 'rear_camera_mp', 'battery_mah']
corr_cols = [c for c in corr_cols if c in df.columns]
plt.figure(figsize=(10, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Корреляционная матрица признаков')
plt.tight_layout()

## 9. Смартфоны: цена vs состояние

In [ ]:
phones = df[df['device_type'] == 'smartphone']
order = [o for o in ['excellent', 'good', 'fair', 'poor'] if o in phones['condition'].unique()]
sns.boxplot(data=phones, x='condition', y='price', order=order)
plt.title('Цена смартфона по состоянию')
plt.tight_layout()

## 10. Ноутбуки: цена vs CPU/GPU

In [ ]:
laptops = df[df['device_type'] == 'laptop']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=laptops, x='cpu_brand', y='price', ax=axes[0])
axes[0].set_title('Цена ноутбука по бренду CPU')
sns.boxplot(data=laptops, x='gpu_brand', y='price', ax=axes[1])
axes[1].set_title('Цена ноутбука по бренду GPU')
plt.tight_layout()

## Выводы EDA

- Цена смартфонов: 8k–150k руб., ноутбуков: 30k–260k руб.
- Наиболее коррелируют с ценой: `storage_gb`, `ram_gb`, `is_laptop`
- Бренд сильно влияет на цену — нужно включать в модель
- Состояние устройства значимо для смартфонов
- Распределение цены правостороннее → логарифмирование улучшает работу линейных моделей